# Hands-on walkthrough
Here we demonstrate the `petropandas` package. For simplicity, we will use star import.

In [ ]:
from petropandas import *

`petropandas` are designed as `pandas.DataFrame` accessors, so we need to create `pandas.DataFrame` before we can use it. There are several accessors available:

| Accessors | Description |
| --- | --- |
| **oxides**     | common calculations with oxides |
| **moles**      | convert to moles |
| **apfu**       | convert to cations |
| **mineral**    | working with mineral models |
| **bulk**       | working with bulk-rock composition |

Here we're using data from built-in examples, but you can read data from files, clipboard etc. For more details, check https://pandas.pydata.org/docs/reference/io.html

In [ ]:
from petropandas.data import minerals as df
from petropandas.data import grt_profile as pro
from petropandas.data import bulk as rock

## Basic usage

Let's see our dataframe...

In [ ]:
df

The accessors `oxides`, `moles` and `apfu` are callable, and by default they return cleaned data.

In [ ]:
df.oxides()

In [ ]:
df.moles()

The accessor `oxides` provides several common utilities like `sorted`, `normalized`, `mean`, `split_valence`, `oxidize`, `reduce` or `select`. Here, we use the `select` to quickly select data of interest. Of course, any other standard `pandas` techniques could be used as well.

Let's select analyses of garnets identified by column `"Mineral"`.

In [ ]:
g = df.oxides.select("Garnet", on="Mineral")
g = g.oxides.sorted()
g

Analyses could be converted to cations p.f.u, either providing number of oxygens:

In [ ]:
g.apfu(n_oxygens=12)

or number of cations:

In [ ]:
g.apfu(n_cations=8)

The `split_valence` method could be used to split low and high valency for selected elements e.g. Fe³⁺/Fe²⁺ using Droop (1987) and Schumacher (1991) methods.

In [ ]:
g.oxides.split_valence("Fe", "droop", 12, 8)

In [ ]:
g.oxides.split_valence("Fe", "schumacher", 12, 8)

## Mineral recalculations
There are several minerals available to work with in petropandas using `mineral` accessor. To calculate mineral apfu from site allocations, excluding remainder, we can use `apfu` method.

In [ ]:
g.mineral.apfu(Grt)

The site allocation could be checked with `site_allocations` method.

In [ ]:
g.mineral.site_allocations(Grt)

and the end-members could be calculated with `end_members` method:

In [ ]:
g.mineral.end_members(Grt)

Similarily, we can recalculate analyses of other minerals.

In [ ]:
df.oxides.select("Muscovite", on="Mineral").mineral.end_members(Ms)

In [ ]:
df.oxides.select("Biotite", on="Mineral").mineral.end_members(Bt)

In [ ]:
df.oxides.select("Amphibole", on="Mineral").mineral.end_members(Amp)

In [ ]:
df.oxides.select("Staurolite", on="Mineral").mineral.end_members(St)

In [ ]:
df.oxides.select("Cordierite", on="Mineral").mineral.end_members(Crd)

## Mineral plots

The `petropandas` has built-in simple plotting system. For profiles you can use `ProfilePlot`. See documentation for options.

In [ ]:
pro

In [ ]:
p = ProfilePlot(columns=["Alm", "Prp", "Grs", "Sps"], split="auto", title="Garnet profile")
p.add(pro.mineral.end_members(Grt), lw=2)
p.show()

We can also plot this data into ternary diagram:

In [ ]:
t = TernaryPlot(top="Prp+Sps", left="Alm", right="Grs", llim=(35,100), tlim=(15,50), title="Garnet profile")
t.add(pro.mineral.end_members(Grt), label="Grt profile")
t.add(g.mineral.end_members(Grt), label="Matrix")
t.show()

Example of AFM diagram. Note chaining of methods `.oxides.reduce().oxides.apatite_correction().moles.normalized()` to obtain correct mol%

In [ ]:
t = TernaryPlot(top="Al2O3-3*K2O-CaO", left="FeO", right="MgO", title="AFM")
for mineral in ['Garnet', 'Biotite', 'Plagioclase', 'Muscovite', 'K-feldspar', 'Chloritoid', 'Cordierite', 'Staurolite']:
    t.add(df.oxides.select(mineral, on="Mineral").oxides.reduce().oxides.apatite_correction().moles.normalized(), label=mineral, marker="x")

t.show()

Scatter plots are also available. Note when using apfu, the ions names in expressions must be back-quoted.

In [ ]:
pairs = [
    ('Garnet', Grt),
    ('Biotite', Bt),
    ('Plagioclase', Fsp),
    ('Muscovite', Ms),
    ('K-feldspar', Fsp),
    ('Chloritoid', Cld),
    ('Cordierite', Crd),
    ('Staurolite', St)
]
s = ScatterPlot("`Si{4+}`/`Al{3+}`", "(`Ca{2+}`+`Na{+}`+`K{+}`)/(`Fe{2+}`+`Fe{3+}`+`Mg{2+}`)", title="Pearce")
for mineral, model in pairs:
    s.add(df.oxides.select(mineral, on="Mineral").mineral.apfu(model), label=mineral)
s.show()

## EPMA bulk analyses

Here we use example data of local bulk rock composition measured from thinsection. To obtain average composition, several areas of the same size have been measured:

In [ ]:
rock

To calculate the average composition, we can calculate the average using `mean`

In [ ]:
avg = rock.oxides.mean()
avg

To use this composition for modelling software, you can use methods to format it. Note that apatite correction and iron conversion to FeO is done automatically. For THERMOCALC

In [ ]:
avg.bulk.TCbulk(H2O=1.5, oxygen=0.1)

for PerpleX

In [ ]:
avg.bulk.Perplexbulk(H2O=1.5, oxygen=0.1)

or MAGEMin

In [ ]:
avg.bulk.MAGEMin(H2O=1.5, oxygen=0.1, db='mp')